In [1]:
import numpy as np
import os
from os import walk
import datetime
import collections
from os.path import exists, getsize, join
from sklearn.model_selection import train_test_split
from scipy import stats
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import random
import math
import time
import sys
import datetime
import numpy as np
import pandas as pd
import os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import classification_report, confusion_matrix

print("✓ Libraries imported successfully")

import torch
print("CUDA available:", torch.cuda.is_available())

ok = hasattr(torch, "version") and getattr(torch.version, "cuda", None) is not None
print("Torch CUDA build info ok:", ok)

if torch.cuda.is_available() and ok:
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("Skip GPU name check due to torch build issue")


INPUT_TRAFFIC_PATH = '/kaggle/input/datasets/hphglinh/mac-new/50labels500_mac_new' 
OUTPUT_PATH = '/kaggle/working/output/'  

MAX_LABELS = 50
MAX_PACKETS = 5000

✓ Libraries imported successfully
CUDA available: True
Torch CUDA build info ok: True
Device: Tesla T4


In [2]:
def extract_label_from_filename(filename):
    # Remove .txt extension
    name = filename.replace('.txt', '')
    
    # Split by '_' and find the last numeric part
    parts = name.split('_')
    
    label_parts = []
    for i, part in enumerate(parts):
        if part.isdigit():
            label_parts = parts[:i]
            break
    else:
        label_parts = parts
    
    # Strip "traffic_" prefix if present
    label = '_'.join(label_parts)
    if label.startswith('traffic_'):
        label = label[8:]
    
    return label

def scan_traffic_files(input_path):    
    if not os.path.exists(input_path):
        print(f"❌ Path not found: {input_path}")
        return {}, {}
    
    all_files = [f for f in os.listdir(input_path) if f.endswith('.txt')]
    
    # Extract label for each file
    file_label_map = {}
    label_set = set()
    for filename in all_files:
        label = extract_label_from_filename(filename)
        label_set.add(label)
        file_label_map[filename] = label
    
    sorted_labels = sorted(label_set)
    
    if len(sorted_labels) > MAX_LABELS:
        print(f"⚠️  {len(sorted_labels)} labels found — limiting to first {MAX_LABELS}")
        sorted_labels = sorted_labels[:MAX_LABELS]
    
    label_mapping = {label: idx for idx, label in enumerate(sorted_labels)}
    
    # Keep only files whose label is within the limit
    filtered_file_label_map = {
        f: l for f, l in file_label_map.items() if l in label_mapping
    }
    
    print(f"📁 {len(all_files)} files found | {len(label_mapping)} labels | {len(filtered_file_label_map)} files to process")
    for label, idx in label_mapping.items():
        count = sum(1 for l in filtered_file_label_map.values() if l == label)
        print(f"  {idx:3d}: {label} ({count} files)")
    
    return label_mapping, filtered_file_label_map

In [3]:
def get_local_ip(path):
    ip_list = []
    with open(path, 'r') as file:
        packets = file.readlines()

    for packet_line in packets:
        packet = packet_line.strip()
        if not packet:
            continue
        strs = packet.split(',')
        if len(strs) < 6:
            continue
        timestamp, src_ip, sport, dst_ip, dport, packet_size = strs[:6]
        ip_list.append(src_ip)
        ip_list.append(dst_ip)

    counter = collections.Counter(ip_list)
    if len(counter) == 0:
        return -1
    local_ip = counter.most_common(1)[0][0]
    return local_ip

print("✓ get_local_ip() defined")

✓ get_local_ip() defined


In [4]:
def parse_pcap(path):
    local_ip = get_local_ip(path)
    all_packets = []
    
    if local_ip == -1:
        return all_packets

    with open(path, 'r') as file:
        packets = file.readlines()

    for packet_line in packets:
        packet = packet_line.strip()
        if not packet:
            continue
        strs = packet.split(',')
        if len(strs) < 6:
            continue
            
        timestamp, src_ip, sport, dst_ip, dport, packet_size = strs[:6]
        arrival_time = datetime.datetime.fromtimestamp(float(timestamp))
        
        sport = int(sport)
        dport = int(dport)
        length = int(packet_size)
        
        # 0 = outgoing, 1 = incoming
        direction = 0 if src_ip == local_ip else 1
        all_packets.append([arrival_time, direction, length])
    
    return all_packets


def filter_valid_files(input_traffic_path, file_label_map, min_packets=100):
    valid_files = set()
    stats = {'too_small': 0, 'valid': 0, 'error': 0}
    
    for filename in file_label_map:
        traffic_file_path = join(input_traffic_path, filename)
        
        if not exists(traffic_file_path) or getsize(traffic_file_path) == 0:
            stats['error'] += 1
            continue
        
        try:
            packet_count = len(parse_pcap(traffic_file_path))
            if packet_count < min_packets:
                stats['too_small'] += 1
            else:
                valid_files.add(filename)
                stats['valid'] += 1
        except Exception:
            stats['error'] += 1
    
    print(f"File filter (min {min_packets} packets): {stats['valid']} valid | {stats['too_small']} too small | {stats['error']} errors")
    return valid_files


def generate_raw_sample(traffic_file_path, label_id, max_packets, min_packets=100):
    if not exists(traffic_file_path) or getsize(traffic_file_path) == 0:
        return None, 0
        
    try:
        all_packets = parse_pcap(traffic_file_path)
        if len(all_packets) < min_packets:
            return None, 0
        
        start_time = all_packets[0][0]
        
        # Columns: [label, relative_time, direction, packet_size] — zero-padded
        raw_data = np.zeros((max_packets, 4))
        for i, pkt in enumerate(all_packets[:max_packets]):
            raw_data[i] = [label_id, (pkt[0] - start_time).total_seconds(), pkt[1], pkt[2]]
        
        return raw_data.flatten().tolist(), len(all_packets)
        
    except Exception as e:
        print(f'Error processing {traffic_file_path}: {e}')
        return None, 0


def process_traffic_data(input_traffic_path, output_path, split_ratio=0.5, random_seed=42):
    os.makedirs(output_path, exist_ok=True)
    embedding_output = join(output_path, 'embedding_data')
    test_output = join(output_path, 'test_data')
    os.makedirs(embedding_output, exist_ok=True)
    os.makedirs(test_output, exist_ok=True)

    max_packets = MAX_PACKETS
    print(f"max_packets={max_packets} | split={split_ratio*100:.0f}/{(1-split_ratio)*100:.0f}")

    label_mapping, file_label_map = scan_traffic_files(input_traffic_path)
    if not label_mapping:
        print("❌ No labels found!")
        return
    
    valid_files = filter_valid_files(input_traffic_path, file_label_map)
    if not valid_files:
        print("❌ No valid files found!")
        return
    
    # Group files by label
    files_by_label = {}
    for filename, label_name in file_label_map.items():
        if filename in valid_files:
            files_by_label.setdefault(label_name, []).append(filename)
    
    # Filter labels with too few files
    MIN_FILES_PER_LABEL = 50
    filtered_labels = {l: f for l, f in files_by_label.items() if len(f) >= MIN_FILES_PER_LABEL}
    removed = {l: len(f) for l, f in files_by_label.items() if len(f) < MIN_FILES_PER_LABEL}
    
    print(f"Label filter (min {MIN_FILES_PER_LABEL} files): {len(filtered_labels)} valid | {len(removed)} removed")
    if removed:
        for label_name, count in sorted(removed.items()):
            print(f"  removed: {label_name} ({count} files)")

    if not filtered_labels:
        print(f"❌ No labels with >= {MIN_FILES_PER_LABEL} files!")
        return
    
    # Rebuild label mapping and save
    new_label_mapping = {label: i for i, label in enumerate(sorted(filtered_labels))}
    label_df = pd.DataFrame(list(new_label_mapping.items()), columns=['label_name', 'label_id'])
    label_df.to_csv(join(embedding_output, 'label_mapping.csv'), index=False)
    label_df.to_csv(join(test_output, 'label_mapping.csv'), index=False)
    
    embedding_samples, test_samples, split_stats = [], [], []
    
    for label_name in sorted(filtered_labels):
        label_id = new_label_mapping[label_name]
        files = sorted(filtered_labels[label_name])
        
        np.random.seed(random_seed)
        shuffled = np.random.permutation(files)
        split_point = int(len(shuffled) * split_ratio)
        embedding_files, test_files = shuffled[:split_point], shuffled[split_point:]
        
        emb_count = trunc_count = 0
        for filename in embedding_files:
            sample, n_pkts = generate_raw_sample(join(input_traffic_path, filename), label_id, max_packets)
            if sample is not None:
                embedding_samples.append(sample)
                emb_count += 1
                if n_pkts > max_packets:
                    trunc_count += 1
        
        test_count = 0
        for filename in test_files:
            sample, n_pkts = generate_raw_sample(join(input_traffic_path, filename), label_id, max_packets)
            if sample is not None:
                test_samples.append(sample)
                test_count += 1
                if n_pkts > max_packets:
                    trunc_count += 1
        
        print(f"  {label_name} (id={label_id}): emb={emb_count} | test={test_count} | truncated={trunc_count}")
        split_stats.append({
            'label_name': label_name, 'label_id': label_id,
            'total_files': len(files),
            'embedding_samples': emb_count, 'test_samples': test_count, 'truncated': trunc_count,
        })
    
    # Save outputs
    for samples, folder, name in [
        (embedding_samples, embedding_output, 'embedding'),
        (test_samples, test_output, 'test'),
    ]:
        if samples:
            pd.DataFrame(samples).to_csv(join(folder, 'raw_packet_data.csv'), header=False, index=False)
        else:
            print(f"❌ No {name} samples generated!")
    
    pd.DataFrame(split_stats).to_csv(join(output_path, 'split_statistics.csv'), index=False)
    print(f"\nDone — {len(filtered_labels)} labels | embedding={len(embedding_samples):,} | test={len(test_samples):,}")
    print(pd.DataFrame(split_stats).to_string(index=False))


process_traffic_data(INPUT_TRAFFIC_PATH, OUTPUT_PATH, split_ratio=0.8, random_seed=42)

max_packets=5000 | split=80/20
📁 20316 files found | 47 labels | 20316 files to process
    0:  (500 files)
    1: acuvue_advanced_for_astigmatism (500 files)
    2: adultplaytoysstoriesinindiana (499 files)
    3: americanbanknainfo (499 files)
    4: arcade_amusement_in_okeechobee (500 files)
    5: automotive_lacquer_paint_for_sale (499 files)
    6: basic_training (500 files)
    7: bible_poems_about_friendship (274 files)
    8: blender_recipes (302 files)
    9: blue_mountain_art_blue_mountain_ny (500 files)
   10: briggs_and_stratton_wiring_diagram (500 files)
   11: celtic_wedding_gown (131 files)
   12: charleskeolhofer (499 files)
   13: chicken_breast_meals (462 files)
   14: cloze_passages (500 files)
   15: dance_clubs (500 files)
   16: departments_jobs (500 files)
   17: disarmament_in_world_war_ii (500 files)
   18: free_downloads_virus_clean_up_without_sercituy_gaurd (99 files)
   19: ftdna (96 files)
   20: gabon_johnson_killed_in_durham_nc_march (500 files)
   21: ho

In [5]:
"""
Author: Yonglong Tian (yonglong@mit.edu)
Date: May 07, 2020
"""
from __future__ import print_function

import torch
import torch.nn as nn
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print(nn.Linear(10, 5))


class SupConLoss(nn.Module):
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    It also supports the unsupervised contrastive loss in SimCLR"""
    def __init__(self, temperature=0.1, contrast_mode='all',
                 base_temperature=0.1):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """Compute loss for model. If both `labels` and `mask` are None,
        it degenerates to SimCLR unsupervised loss:
        https://arxiv.org/pdf/2002.05709.pdf

        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """
        device = (torch.device('cuda')
                  if features.is_cuda
                  else torch.device('cpu'))

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        # modified to handle edge cases when there is no positive pair
        # for an anchor point. 
        # Edge case e.g.:- 
        # features of shape: [4,1,...]
        # labels:            [0,1,1,2]
        # loss before mean:  [nan, ..., ..., nan] 
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

torch: 2.9.0+cu126
cuda available: True
Linear(in_features=10, out_features=5, bias=True)


In [6]:
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    RAW_DATA_PATH = '/kaggle/working/output/embedding_data/raw_packet_data.csv'
    LABEL_MAPPING_PATH = '/kaggle/working/output/embedding_data/label_mapping.csv'
    MODEL_PATH = '/kaggle/working/raw_packet_embedding_supcon.pth'

# Training parameters
BATCH_SIZE = 32
LEARNING_RATE = 0.1
WEIGHT_DECAY = 1e-4
MOMENTUM = 0.9
EPOCHS = 500
TEMPERATURE = 0.1

# Learning rate schedule
LR_DECAY_EPOCHS = [700, 800, 900]
LR_DECAY_RATE = 0.1
COSINE_SCHEDULE = True
WARMUP = True
WARMUP_EPOCHS = 10
WARMUP_FROM = 0.01

# Model parameters
HIDDEN_SIZE = 256
EMBEDDING_SIZE = 128

# Data parameters
MIN_PACKETS = 100
MAX_PACKETS = 25000

print(f"Running on: {'KAGGLE' if IS_KAGGLE else 'LOCAL'}")
print(f"Raw data path: {RAW_DATA_PATH}")

Running on: KAGGLE
Raw data path: /kaggle/working/output/embedding_data/raw_packet_data.csv


In [7]:
class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def adjust_learning_rate(optimizer, epoch):
    lr = LEARNING_RATE
    if COSINE_SCHEDULE:
        eta_min = lr * (LR_DECAY_RATE ** 3)
        lr = eta_min + (lr - eta_min) * (1 + math.cos(math.pi * epoch / EPOCHS)) / 2
    else:
        steps = np.sum(epoch > np.asarray(LR_DECAY_EPOCHS))
        if steps > 0:
            lr = lr * (LR_DECAY_RATE ** steps)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    return lr


def warmup_learning_rate(optimizer, epoch, batch_id, total_batches):
    if not WARMUP or epoch > WARMUP_EPOCHS:
        return
    p = (batch_id + (epoch - 1) * total_batches) / (WARMUP_EPOCHS * total_batches)
    if COSINE_SCHEDULE:
        eta_min = LEARNING_RATE * (LR_DECAY_RATE ** 3)
        warmup_to = eta_min + (LEARNING_RATE - eta_min) * (
            1 + math.cos(math.pi * WARMUP_EPOCHS / EPOCHS)) / 2
    else:
        warmup_to = LEARNING_RATE
    lr = WARMUP_FROM + p * (warmup_to - WARMUP_FROM)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr


def save_model(model, optimizer, epoch, save_path):
    state = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'model_config': {
            'max_packets': model.max_packets if hasattr(model, 'max_packets') else MAX_PACKETS,
            'hidden_size': HIDDEN_SIZE,
            'embedding_size': EMBEDDING_SIZE,
        }
    }
    torch.save(state, save_path)
    print(f'Model saved to {save_path}')


# ============================================================================
# DATA PROCESSING
# ============================================================================

def load_and_process_raw_data(raw_data_path, label_mapping_path):
    """Load raw packet data and reshape to (N, max_packets, 3)"""
    label_names = {}
    if os.path.exists(label_mapping_path):
        label_df = pd.read_csv(label_mapping_path)
        label_names = dict(zip(label_df['label_id'], label_df['label_name']))
    
    raw_df = pd.read_csv(raw_data_path, header=None)
    max_packets = len(raw_df.columns) // 4

    all_samples, all_labels = [], []
    for _, row in raw_df.iterrows():
        sample = row.values.reshape(max_packets, 4)
        all_labels.append(int(sample[0, 0]))
        all_samples.append(sample[:, 1:])  # keep [time, direction, size]

    X = np.array(all_samples, dtype=np.float32)
    y = np.array(all_labels, dtype=np.int64)

    print(f"Loaded {len(X)} samples | {len(np.unique(y))} labels | shape per sample: ({max_packets}, 3)")
    return X, y, label_names, max_packets


def organize_data_by_label(X, y, samples_per_class=250):
    data_table = {}
    for sample, label in zip(X, y):
        data_table.setdefault(label, []).append(sample)

    data_list, filtered_labels = [], []
    for label, samples in data_table.items():
        if len(samples) > 1:
            if len(samples) > samples_per_class:
                samples = random.sample(samples, samples_per_class)
            data_list.append(samples)
            filtered_labels.append(label)

    total = sum(len(s) for s in data_list)
    print(f"Organized: {len(data_list)} labels | {total} samples | max {samples_per_class}/class")
    return data_list, filtered_labels


# ============================================================================
# MODEL
# ============================================================================

class SFE(nn.Module):
    def __init__(self, max_packets, hidden_size=256, embedding_size=128):
        super(SFE, self).__init__()
        self.max_packets = max_packets
        self.hidden_size = hidden_size
        self.embedding_size = embedding_size

        kernel_size = 8
        conv_stride = 1
        pool_stride = 4
        pool_size = 8

        self.conv1   = nn.Conv1d(3,   32,  kernel_size, stride=conv_stride)
        self.conv1_1 = nn.Conv1d(32,  32,  kernel_size, stride=conv_stride)
        self.conv2   = nn.Conv1d(32,  64,  kernel_size, stride=conv_stride)
        self.conv2_2 = nn.Conv1d(64,  64,  kernel_size, stride=conv_stride)
        self.conv3   = nn.Conv1d(64,  128, kernel_size, stride=conv_stride)
        self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride=conv_stride)
        self.conv4   = nn.Conv1d(128, 256, kernel_size, stride=conv_stride)
        self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride=conv_stride)

        self.batch_norm1 = nn.BatchNorm1d(32)
        self.batch_norm2 = nn.BatchNorm1d(64)
        self.batch_norm3 = nn.BatchNorm1d(128)
        self.batch_norm4 = nn.BatchNorm1d(256)

        self.max_pool_1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_2 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_3 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_4 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)

        self.dropout1 = nn.Dropout(p=0.1)
        self.dropout2 = nn.Dropout(p=0.1)
        self.dropout3 = nn.Dropout(p=0.1)
        self.dropout4 = nn.Dropout(p=0.1)

        with torch.no_grad():
            dummy = torch.zeros(1, 3, max_packets)
            flat_dim = self._forward_convs(dummy).view(1, -1).shape[1]

        self.fc = nn.Linear(flat_dim, hidden_size)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _forward_convs(self, x):
        # Block 1 (ELU)
        x = F.pad(x, (3, 4)); x = F.elu(self.conv1(x))
        x = F.pad(x, (3, 4)); x = F.elu(self.batch_norm1(self.conv1_1(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_1(x)
        x = self.dropout1(x)
        # Block 2 (ReLU)
        x = F.pad(x, (3, 4)); x = F.relu(self.conv2(x))
        x = F.pad(x, (3, 4)); x = F.relu(self.batch_norm2(self.conv2_2(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_2(x)
        x = self.dropout2(x)
        # Block 3 (ReLU)
        x = F.pad(x, (3, 4)); x = F.relu(self.conv3(x))
        x = F.pad(x, (3, 4)); x = F.relu(self.batch_norm3(self.conv3_3(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_3(x)
        x = self.dropout3(x)
        # Block 4 (ReLU)
        x = F.pad(x, (3, 4)); x = F.relu(self.conv4(x))
        x = F.pad(x, (3, 4)); x = F.relu(self.batch_norm4(self.conv4_4(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_4(x)
        x = self.dropout4(x)
        return x

    def forward(self, x):
        # x: (batch, max_packets, 3)
        x = x.transpose(1, 2)      # -> (batch, 3, max_packets)
        x = self._forward_convs(x)
        x = x.view(x.size(0), -1)  # flatten
        return self.fc(x)           # -> (batch, hidden_size)


class SupconNet(nn.Module):
    def __init__(self, max_packets, hidden_size=256, embedding_size=128, head='mlp'):
        super(SupconNet, self).__init__()
        self.max_packets = max_packets
        self.encoder = SFE(max_packets, hidden_size, embedding_size)

        dim_mlp = hidden_size
        if head == 'linear':
            self.head = nn.Linear(dim_mlp, embedding_size)
        elif head == 'mlp':
            self.head = nn.Sequential(
                nn.Linear(dim_mlp, dim_mlp),
                nn.BatchNorm1d(dim_mlp),
                nn.ReLU(inplace=True),
                nn.Linear(dim_mlp, embedding_size)
            )
        else:
            raise NotImplementedError(f'head not supported: {head}')

    def forward(self, x):
        feat = self.encoder(x)
        return F.normalize(self.head(feat), dim=1)


# ============================================================================
# DATA AUGMENTATION
# ============================================================================

def NetFlowAugmnet(x):
    """Random augmentation for packet sequences."""
    augmented = x.clone()

    # Time jitter
    if torch.rand(1).item() > 0.2:
        time_col = augmented[:, 0]
        jitter = (torch.rand_like(time_col) - 0.5) * 0.2
        augmented[:, 0] = torch.clamp(time_col + jitter, min=0)

    # Size scaling
    if torch.rand(1).item() > 0.2:
        scale = 0.9 + torch.rand(1).item() * 0.2
        augmented[:, 2] = augmented[:, 2] * scale

    # Packet dropout
    if torch.rand(1).item() > 0.5:
        mask = (torch.rand(x.shape[0]) > 0.1).unsqueeze(1).expand_as(x).to(x.device)
        augmented = augmented * mask.float()

    # Gaussian noise
    if torch.rand(1).item() > 0.5:
        augmented = augmented + torch.randn_like(augmented) * 0.05

    return torch.nan_to_num(augmented, nan=0.0, posinf=0.0, neginf=0.0)


class TwoCropTransform:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, x):
        return [self.transform(x), self.transform(x)]


# ============================================================================
# DATASET
# ============================================================================

class PacketDataset(torch.utils.data.Dataset):
    """Packet sequence dataset with two-view augmentation."""
    def __init__(self, data_list, transform=None):
        self.transform = transform
        self.samples, self.labels = [], []

        for label_idx, samples in enumerate(data_list):
            for sample in samples:
                if isinstance(sample, np.ndarray):
                    sample = torch.from_numpy(sample.astype(np.float32))
                self.samples.append(sample)
                self.labels.append(label_idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        label = self.labels[idx]
        if self.transform is not None:
            return self.transform(sample), label
        return sample, label


# ============================================================================
# TRAINING
# ============================================================================

def train(train_loader, model, criterion, optimizer, epoch):
    model.train()
    losses = AverageMeter()

    for idx, (images, labels) in enumerate(train_loader):
        images = torch.cat([images[0], images[1]], dim=0)
        if torch.cuda.is_available():
            images = images.cuda(non_blocking=True)
            labels = labels.cuda(non_blocking=True)

        bsz = labels.shape[0]
        warmup_learning_rate(optimizer, epoch, idx, len(train_loader))

        features = model(images)
        f1, f2 = torch.split(features, [bsz, bsz], dim=0)
        features = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)
        loss = criterion(features, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  WARNING: NaN/Inf loss at batch {idx}, skipping")
            continue

        losses.update(loss.item(), bsz)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return losses.avg


# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================

def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Device: {device}")

    X, y, label_names, max_packets = load_and_process_raw_data(RAW_DATA_PATH, LABEL_MAPPING_PATH)
    data_list, filtered_labels = organize_data_by_label(X, y)

    if len(data_list) < 2:
        print("❌ Need at least 2 classes!")
        return False

    transform = TwoCropTransform(NetFlowAugmnet)
    train_dataset = PacketDataset(data_list, transform=transform)
    actual_batch_size = min(BATCH_SIZE, len(train_dataset))

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=actual_batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        drop_last=(len(train_dataset) > actual_batch_size)
    )

    if len(train_loader) == 0:
        print("❌ train_loader is empty!")
        return False

    model = SupconNet(max_packets=max_packets, hidden_size=HIDDEN_SIZE,
                      embedding_size=EMBEDDING_SIZE, head='mlp')
    criterion = SupConLoss(temperature=TEMPERATURE)

    if torch.cuda.is_available():
        model = model.cuda()
        criterion = criterion.cuda()

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model params: {total_params:,} | batch={actual_batch_size} | lr={LEARNING_RATE} | epochs={EPOCHS}")

    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        adjust_learning_rate(optimizer, epoch)
        loss = train(train_loader, model, criterion, optimizer, epoch)
        if epoch % 50 == 0:
            print(f"Epoch {epoch}/{EPOCHS} | loss={loss:.4f}")

    save_model(model, optimizer, EPOCHS, MODEL_PATH)
    return True


if __name__ == '__main__':
    if not main():
        print("❌ Training failed")
        sys.exit(1)

Device: cuda
Loaded 16135 samples | 46 labels | shape per sample: (5000, 3)
Organized: 46 labels | 10559 samples | max 250/class
Model params: 2,453,248 | batch=32 | lr=0.1 | epochs=500
Epoch 50/500 | loss=0.6979
Epoch 100/500 | loss=0.6766
Epoch 150/500 | loss=0.6898
Epoch 200/500 | loss=0.7051
Epoch 250/500 | loss=0.6684
Epoch 300/500 | loss=0.6491
Epoch 350/500 | loss=0.6491
Epoch 400/500 | loss=0.6599
Epoch 450/500 | loss=0.6417
Epoch 500/500 | loss=0.6625
Model saved to /kaggle/working/raw_packet_embedding_supcon.pth


In [ ]:
# ============================================================================
# CONFIGURATION - CLOSED WORLD
# ============================================================================
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    EMBEDDING_DATA_PATH = '/kaggle/working/output/embedding_data/raw_packet_data.csv'
    TEST_DATA_PATH      = '/kaggle/working/output/test_data/raw_packet_data.csv'
    LABEL_MAPPING_PATH  = '/kaggle/working/output/embedding_data/label_mapping.csv'
    MODEL_PATH          = '/kaggle/working/raw_packet_embedding_supcon.pth'
    OUTPUT_FOLDER       = '/kaggle/working/output'

# Set to None to use all training samples, or an int to cap per class
TRAIN_SAMPLES_PER_CLASS = None


def load_pretrained_embedding_model(model_path):
    if not os.path.exists(model_path):
        print(f"❌ Model file not found: {model_path}")
        return None

    checkpoint = torch.load(model_path, map_location='cpu')
    config = checkpoint.get('model_config', {})

    model = SupconNet(
        max_packets=config.get('max_packets', 25000),
        hidden_size=config.get('hidden_size', 256),
        embedding_size=config.get('embedding_size', 128),
        head='mlp'
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model  = model.to(device)
    print(f"Embedding model loaded on {device} (epoch {checkpoint.get('epoch', '?')})")
    return model


def extract_embeddings_from_raw(model, X_raw, batch_size=32):
    """Extract features after encoder fc, before projection head — shape: (batch, hidden_size)"""
    encoder = model.encoder
    encoder.eval()
    device = next(model.parameters()).device
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(X_raw), batch_size):
            batch_tensor = torch.FloatTensor(X_raw[i:i + batch_size]).to(device)
            embeddings.append(encoder(batch_tensor).cpu().numpy())

    return np.vstack(embeddings)


def subsample_per_class(X, y, samples_per_class, seed=42):
    """Subsample at most `samples_per_class` examples per class."""
    rng = np.random.default_rng(seed)
    indices = []
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        if len(idx) > samples_per_class:
            idx = rng.choice(idx, size=samples_per_class, replace=False)
        indices.append(idx)
    indices = np.concatenate(indices)
    rng.shuffle(indices)
    return X[indices], y[indices]


# ============================================================================
# MLP CLASSIFIER (3 LAYERS)
# ============================================================================

class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


def train_mlp(X_train, y_train, num_classes, input_dim,
              hidden_dim=512, epochs=50, batch_size=256,
              lr=1e-3, dropout=0.3, device='cpu'):
    print(f"Training MLP: input={input_dim} | hidden={hidden_dim} | classes={num_classes} | "
          f"samples={len(X_train)} | epochs={epochs} | lr={lr}")

    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(y_train).to(device)),
        batch_size=batch_size, shuffle=True
    )

    model     = MLPClassifier(input_dim, hidden_dim, num_classes, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)
            correct    += (logits.argmax(1) == yb).sum().item()
            total      += len(xb)

        scheduler.step()
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs} | loss={total_loss/total:.4f} | acc={correct/total:.4f}")

    return model


@torch.no_grad()
def predict_mlp(model, X, batch_size=256, device='cpu'):
    model.eval()
    loader = DataLoader(TensorDataset(torch.FloatTensor(X).to(device)), batch_size=batch_size)
    return np.concatenate([model(xb).argmax(1).cpu().numpy() for (xb,) in loader])


@torch.no_grad()
def predict_proba_mlp(model, X, batch_size=256, device='cpu'):
    """Return softmax probability matrix — shape: (N, num_classes)"""
    model.eval()
    loader = DataLoader(TensorDataset(torch.FloatTensor(X).to(device)), batch_size=batch_size)
    return np.concatenate([
        torch.softmax(model(xb), dim=1).cpu().numpy() for (xb,) in loader
    ])


# ============================================================================
# DATA LOADING
# ============================================================================

def load_raw_packet_data(data_path, label_mapping_path=None, data_type='data'):
    label_names = {}
    if label_mapping_path and os.path.exists(label_mapping_path):
        label_df   = pd.read_csv(label_mapping_path)
        label_names = dict(zip(label_df['label_id'], label_df['label_name']))

    if not os.path.exists(data_path):
        print(f"❌ Data file not found: {data_path}")
        return None, None, label_names

    raw_df      = pd.read_csv(data_path, header=None)
    max_packets = len(raw_df.columns) // 4

    all_samples, all_labels = [], []
    for _, row in raw_df.iterrows():
        sample = row.values.reshape(max_packets, 4)
        all_labels.append(int(sample[0, 0]))
        all_samples.append(sample[:, 1:])

    X = np.array(all_samples, dtype=np.float32)
    y = np.array(all_labels,  dtype=np.int64)

    print(f"{data_type}: {X.shape}, {len(np.unique(y))} labels")
    return X, y, label_names


# ============================================================================
# MAIN - CLOSED WORLD
# ============================================================================

def main_closed_world():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    emb_model = load_pretrained_embedding_model(MODEL_PATH)
    if emb_model is None:
        return

    X_train_raw, y_train, label_names = load_raw_packet_data(EMBEDDING_DATA_PATH, LABEL_MAPPING_PATH, 'train')
    X_test_raw,  y_test,  _           = load_raw_packet_data(TEST_DATA_PATH,       LABEL_MAPPING_PATH, 'test')
    if X_train_raw is None or X_test_raw is None:
        return

    print("Extracting embeddings...")
    X_train_emb = extract_embeddings_from_raw(emb_model, X_train_raw)
    X_test_emb  = extract_embeddings_from_raw(emb_model, X_test_raw)
    print(f"  train={X_train_emb.shape} | test={X_test_emb.shape}")

    # Remap labels to 0-indexed for CrossEntropyLoss
    unique_labels = np.unique(y_train)
    label2idx     = {lbl: i for i, lbl in enumerate(unique_labels)}
    idx2label     = {i: lbl for lbl, i in label2idx.items()}
    y_train_idx   = np.array([label2idx[l] for l in y_train])

    if TRAIN_SAMPLES_PER_CLASS is not None:
        X_train_emb, y_train_idx = subsample_per_class(X_train_emb, y_train_idx, TRAIN_SAMPLES_PER_CLASS)
        print(f"Subsampled to {TRAIN_SAMPLES_PER_CLASS} samples/class -> {len(X_train_emb)} total")

    mlp = train_mlp(
        X_train_emb, y_train_idx,
        num_classes=len(unique_labels),
        input_dim=X_train_emb.shape[1],
        hidden_dim=512, epochs=50, batch_size=256, lr=1e-3, dropout=0.3,
        device=device
    )

    y_pred_idx = predict_mlp(mlp, X_test_emb, device=device)
    y_pred     = np.array([idx2label[i] for i in y_pred_idx])

    unique_all   = np.unique(np.concatenate([y_train, y_test]))
    target_names = [label_names.get(l, f'Class_{l}') for l in unique_all]

    report = classification_report(
        y_test, y_pred,
        labels=unique_all, target_names=[str(x) for x in target_names],
        digits=4, zero_division=0
    )
    print("\nCLASSIFICATION REPORT\n")
    print(report)

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    report_path = os.path.join(OUTPUT_FOLDER, 'classification_report_closed_world.txt')
    with open(report_path, 'w') as f:
        f.write("CLASSIFICATION REPORT - MLP CLOSED WORLD\n\n")
        f.write(report)
    print(f"Report saved: {report_path}")

    mlp_path = os.path.join(OUTPUT_FOLDER, 'mlp_classifier_closed_world.pth')
    torch.save(mlp.state_dict(), mlp_path)
    print(f"MLP weights saved: {mlp_path}")

    return mlp  # return để cell open-world có thể reuse nếu cần


main_closed_world()

Embedding model loaded on cuda (epoch 500)
train: (16135, 5000, 3), 46 labels
test: (4043, 5000, 3), 46 labels
Extracting embeddings...
  train=(16135, 256) | test=(4043, 256)
Training MLP: input=256 | hidden=512 | classes=46 | samples=16135 | epochs=50 | lr=0.001
  Epoch   1/50 | loss=0.5330 | acc=0.9540


In [ ]:
# ============================================================================
# CONFIGURATION - OPEN WORLD
# ============================================================================

KNOWN_TRAIN_PATH  = 'data_pretrain/50labels500_mac15000/embedding_data/raw_packet_data.csv'
KNOWN_TEST_PATH   = 'data_pretrain/50labels500_mac15000/test_data/raw_packet_data.csv'
UNKNOWN_DATA_PATH = 'data_pretrain/50labels500_mac_new/embedding_data/raw_packet_data.csv'
OW_LABEL_MAPPING_PATH = 'data_pretrain/50labels500_mac/embedding_data/label_mapping.csv'
OW_MODEL_PATH     = 'model/augment/50labels500_mac_2n2.pth'
OW_OUTPUT_FOLDER  = 'output/ow_mlp'

# ID reserved for the unknown class (must be outside the known label range)
UNKNOWN_CLASS_ID      = 50
UNKNOWN_TRAIN_SAMPLES = 400
UNKNOWN_TEST_SAMPLES  = 10000   # None = all remaining

# MLP hyperparams
OW_HIDDEN_DIM  = 512
OW_EPOCHS      = 50
OW_BATCH_SIZE  = 256
OW_LR          = 1e-3
OW_DROPOUT     = 0.3

# ============================================================================
# HELPERS
# ============================================================================

def load_raw_packet_data_ow(data_path, data_type='data', read_labels=True):
    """Load CSV into (X, y). y=None if read_labels=False."""
    if not os.path.exists(data_path):
        print(f"❌ Not found: {data_path}")
        return None, None

    raw_df      = pd.read_csv(data_path, header=None)
    max_packets = len(raw_df.columns) // 4

    samples, labels = [], []
    for _, row in raw_df.iterrows():
        sample = row.values.reshape(max_packets, 4)
        if read_labels:
            labels.append(int(sample[0, 0]))
        samples.append(sample[:, 1:])

    X = np.array(samples, dtype=np.float32)
    y = np.array(labels,  dtype=np.int64) if read_labels else None
    print(f"{data_type}: {X.shape}" + (f", {len(np.unique(y))} labels" if y is not None else ""))
    return X, y


def load_unknown_pool(unknown_path, train_n, test_n, unknown_class_id):
    """
    Load unknown pool, shuffle, split into non-overlapping train/test partitions.
    test_n=None uses all remaining samples after train_n.
    """
    X_unk, _ = load_raw_packet_data_ow(unknown_path, data_type='unknown pool', read_labels=False)
    if X_unk is None:
        return None, None, None, None

    rng  = np.random.default_rng(seed=42)
    X_unk = X_unk[rng.permutation(len(X_unk))]

    total         = len(X_unk)
    test_n_actual = (total - train_n) if test_n is None else test_n
    need          = train_n + test_n_actual

    if need > total:
        print(f"  ⚠️  Requested {need} unknown samples but only {total} available — adjusting.")
        train_n       = min(train_n, total)
        test_n_actual = total - train_n

    X_unk_train = X_unk[:train_n]
    X_unk_test  = X_unk[train_n:train_n + test_n_actual]
    y_unk_train = np.full(len(X_unk_train), unknown_class_id, dtype=np.int64)
    y_unk_test  = np.full(len(X_unk_test),  unknown_class_id, dtype=np.int64)

    print(f"  unknown train={len(y_unk_train)} | unknown test={len(y_unk_test)}")
    return X_unk_train, y_unk_train, X_unk_test, y_unk_test


def build_label_names_ow(y_known, label_mapping_path, unknown_class_id):
    id_to_name = {int(cid): f"class_{cid}" for cid in np.unique(y_known)}
    id_to_name[unknown_class_id] = "Unknown"

    if label_mapping_path and os.path.exists(label_mapping_path):
        lmap = pd.read_csv(label_mapping_path)
        if {'label_id', 'label_name'}.issubset(lmap.columns):
            for _, row in lmap.iterrows():
                cid = int(row['label_id'])
                if cid in id_to_name:
                    id_to_name[cid] = str(row['label_name'])
    return id_to_name


# ============================================================================
# MAIN - OPEN WORLD
# ============================================================================

def main_open_world():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Load embedding model
    emb_model = load_pretrained_embedding_model(OW_MODEL_PATH)
    if emb_model is None:
        return

    # Load known splits
    X_known_train, y_known_train = load_raw_packet_data_ow(KNOWN_TRAIN_PATH, 'known train')
    X_known_test,  y_known_test  = load_raw_packet_data_ow(KNOWN_TEST_PATH,  'known test')
    if X_known_train is None or X_known_test is None:
        return

    # Load & split unknown pool
    X_unk_train, y_unk_train, X_unk_test, y_unk_test = load_unknown_pool(
        UNKNOWN_DATA_PATH, UNKNOWN_TRAIN_SAMPLES, UNKNOWN_TEST_SAMPLES, UNKNOWN_CLASS_ID
    )
    if X_unk_train is None:
        return

    # Merge known + unknown
    rng = np.random.default_rng(seed=0)

    X_train_raw = np.concatenate([X_known_train, X_unk_train])
    y_train     = np.concatenate([y_known_train, y_unk_train])
    perm        = rng.permutation(len(X_train_raw))
    X_train_raw, y_train = X_train_raw[perm], y_train[perm]

    X_test_raw = np.concatenate([X_known_test, X_unk_test])
    y_test     = np.concatenate([y_known_test,  y_unk_test])
    perm       = rng.permutation(len(X_test_raw))
    X_test_raw, y_test = X_test_raw[perm], y_test[perm]

    print(f"Final train={len(y_train)} | test={len(y_test)}")

    # Label names
    id_to_name = build_label_names_ow(
        np.concatenate([y_known_train, y_known_test]),
        OW_LABEL_MAPPING_PATH, UNKNOWN_CLASS_ID
    )

    # Extract embeddings
    print("Extracting embeddings...")
    X_train_emb = extract_embeddings_from_raw(emb_model, X_train_raw)
    X_test_emb  = extract_embeddings_from_raw(emb_model, X_test_raw)
    print(f"  train={X_train_emb.shape} | test={X_test_emb.shape}")

    # Remap all labels (known + unknown class id) to 0-indexed
    unique_labels = np.unique(y_train)
    label2idx     = {lbl: i for i, lbl in enumerate(unique_labels)}
    idx2label     = {i: lbl for lbl, i in label2idx.items()}
    y_train_idx   = np.array([label2idx[l] for l in y_train])
    unknown_idx   = label2idx[UNKNOWN_CLASS_ID]  # internal index of unknown class

    # Train MLP
    mlp = train_mlp(
        X_train_emb, y_train_idx,
        num_classes=len(unique_labels),
        input_dim=X_train_emb.shape[1],
        hidden_dim=OW_HIDDEN_DIM, epochs=OW_EPOCHS,
        batch_size=OW_BATCH_SIZE, lr=OW_LR, dropout=OW_DROPOUT,
        device=device
    )

    # Predict
    y_pred_idx = predict_mlp(mlp, X_test_emb, device=device)
    y_pred     = np.array([idx2label[i] for i in y_pred_idx])
    proba      = predict_proba_mlp(mlp, X_test_emb, device=device)  # (N, num_classes)

    # ── Classification Report ─────────────────────────────────
    all_ids      = sorted(np.unique(np.concatenate([y_train, y_test])))
    target_names = [id_to_name.get(cid, f"class_{cid}") for cid in all_ids]

    report = classification_report(
        y_test, y_pred,
        labels=all_ids, target_names=target_names,
        digits=4, zero_division=0
    )
    print("\nCLASSIFICATION REPORT\n")
    print(report)

    # ── Binary (known vs unknown) confusion matrix ────────────
    y_test_bin = (y_test != UNKNOWN_CLASS_ID).astype(int)
    y_pred_bin = (y_pred != UNKNOWN_CLASS_ID).astype(int)
    cm_bin     = confusion_matrix(y_test_bin, y_pred_bin, labels=[0, 1])
    cm_bin_df  = pd.DataFrame(
        cm_bin,
        index   = ['Actual Unknown', 'Actual Known'],
        columns = ['Pred Unknown',   'Pred Known'],
    )
    print("BINARY CONFUSION MATRIX (Known vs Unknown)")
    print(cm_bin_df.to_string())
    tn, fp, fn, tp = cm_bin.ravel()
    print(f"  TN={tn}  FP={fp}  FN={fn}  TP={tp}")

    # ── Open-world curves (max known-class confidence as score) ──
    known_mask  = np.array([i for i, lbl in idx2label.items() if lbl != UNKNOWN_CLASS_ID])
    score_known = proba[:, known_mask].max(axis=1)
    y_true_bin  = (y_test != UNKNOWN_CLASS_ID).astype(int)

    precision, recall, _ = precision_recall_curve(y_true_bin, score_known)
    fpr, tpr, _          = roc_curve(y_true_bin, score_known)

    # ── Full confusion matrix ─────────────────────────────────
    cm_full    = confusion_matrix(y_test, y_pred, labels=all_ids)
    cm_full_df = pd.DataFrame(cm_full, index=target_names, columns=target_names)

    # ── Save outputs ──────────────────────────────────────────
    os.makedirs(OW_OUTPUT_FOLDER, exist_ok=True)

    report_path = os.path.join(OW_OUTPUT_FOLDER, 'classification_report_open_world.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(f"OPEN-WORLD MLP CLASSIFICATION REPORT\n")
        f.write(f"  Known classes : 0..{UNKNOWN_CLASS_ID-1} | Unknown class: {UNKNOWN_CLASS_ID}\n")
        f.write(f"  Unknown TRAIN={UNKNOWN_TRAIN_SAMPLES} | TEST={UNKNOWN_TEST_SAMPLES}\n\n")
        f.write(report)
        f.write(f"\n\nBINARY CONFUSION MATRIX\n{cm_bin_df.to_string()}\n")
        f.write(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}\n")
    print(f"Report saved: {report_path}")

    cm_full_df.to_csv(os.path.join(OW_OUTPUT_FOLDER, 'confusion_matrix_full.csv'))
    cm_bin_df.to_csv(os.path.join(OW_OUTPUT_FOLDER, 'confusion_matrix_binary.csv'))

    pred_df = pd.DataFrame({
        'y_true':      y_test,
        'y_true_name': [id_to_name.get(v, f"class_{v}") for v in y_test],
        'y_pred':      y_pred,
        'y_pred_name': [id_to_name.get(v, f"class_{v}") for v in y_pred],
    })
    pred_df.to_csv(os.path.join(OW_OUTPUT_FOLDER, 'predictions.csv'), index=False)

    curve_folder = os.path.join(OW_OUTPUT_FOLDER, 'curve_data')
    os.makedirs(curve_folder, exist_ok=True)
    pd.DataFrame({'precision': precision, 'recall': recall}).to_csv(
        os.path.join(curve_folder, 'precision_recall_curve.csv'), index=False)
    pd.DataFrame({'fpr': fpr, 'tpr': tpr}).to_csv(
        os.path.join(curve_folder, 'roc_curve.csv'), index=False)

    mlp_path = os.path.join(OW_OUTPUT_FOLDER, 'mlp_classifier_open_world.pth')
    torch.save(mlp.state_dict(), mlp_path)
    print(f"MLP weights saved: {mlp_path}")


main_open_world()